# Remove Background — self-hosted example

This notebook runs Bria **Remove Background** (RMBG 2.0) on your own GPU: it asks the Bria Engine for a short-lived package credential, installs **`remove_bg`** from Bria's **`bria-remove-bg`** repository, and runs the pipeline on a product image. The model weights are downloaded from Bria on the first `setup()`.

**Prerequisites**

- A Bria API key in `BRIA_API_TOKEN` — any key of an account that holds a self-hosting license for RMBG 2.0 (https://platform.bria.ai/outpost/licenses?model=rmbg-2.0).
- Python 3.10 or newer and an NVIDIA GPU.
- Network access to the Bria Engine, AWS CodeArtifact and Bria's weights mirror (CloudFront). No Hugging Face account.


In [ ]:
%pip install -q python-dotenv

from dotenv import load_dotenv

load_dotenv()  # loads BRIA_API_TOKEN (and friends) from a local .env file, if present


## 1. Package credential

Request a short-lived credential for the **`bria-remove-bg`** PyPI repository. The Engine issues it only for a model your account holds a license for; a `403` with `not_entitled` means the account has no license for RMBG 2.0.


In [ ]:
import json
import os
from urllib.error import HTTPError
from urllib.request import Request, urlopen

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN", "").strip()  # a key from a secret file ends in a newline
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

ENGINE = os.environ.get("BRIA_CLIENT_BASE_URL", "https://engine.prod.bria-api.com")
url = f"{ENGINE}/v2/outpost/access/code_artifact?repository=bria-remove-bg"
request = Request(url, headers={"api_token": BRIA_API_TOKEN})

try:
    with urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))
except HTTPError as exc:
    body = exc.read().decode("utf-8", errors="replace")
    try:
        code = json.loads(body).get("error", {}).get("message")
    except json.JSONDecodeError:
        code = None
    if code in ("not_entitled", "entitlement_revoked"):
        raise RuntimeError(
            "This account has no active license for RMBG 2.0. Get one at https://platform.bria.ai/outpost/licenses?model=rmbg-2.0"
        ) from exc
    raise RuntimeError(f"Package credential request failed with HTTP {exc.code}: {body or exc.reason}") from exc

result = payload.get("result") or {}
CODE_ARTIFACT_PASSWORD = (result.get("authorization_token") or "").strip()
if not CODE_ARTIFACT_PASSWORD:
    raise RuntimeError(f"The response did not include an authorization token: {payload}")
print("Package credential acquired. Expires:", result.get("expiration"))


## 2. Install dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

# pip calls os.getcwd(): if Jupyter's working folder was deleted, move to the home folder first.
try:
    Path.cwd()
except FileNotFoundError:
    os.chdir(Path.home())

encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
remove_bg_index = "https://aws:" + encoded_password + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-remove-bg/simple/"


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(install_cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install([sys.executable, "-m", "pip", "install", "--upgrade", "remove_bg>=0.5.0", "--extra-index-url", remove_bg_index])


## 3. Start the local pipeline

`setup()` first checks your license with the Bria Engine, then downloads the RMBG 2.0 weights from Bria into `~/.bria/models/rmbg-2.0/<version>/` (set `BRIA_HOME` to move them) and verifies them; later runs reuse them. If the license is missing or has ended, `setup()` raises `LicenseError` with a link to your licenses page and loads nothing.

The example disables model compilation for a simpler first run; set `compile_model=True` in a long-lived service. If the install cell changed core runtime packages, restart the kernel before this cell.


In [ ]:
import logging
from pathlib import Path

from common.outpost import LicenseError
from remove_bg import RemoveBG, RemoveBGConfig

logging.basicConfig(level=logging.INFO)

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

remove_bg = RemoveBG(config=RemoveBGConfig(compile_model=False, api_token=BRIA_API_TOKEN))
try:
    remove_bg.setup()
except LicenseError as error:
    raise RuntimeError(f"{error} ({error.code})") from None
print("Remove Background pipeline setup complete.")


## 4. Remove The Background

Pass either a URL or a local image path into `RemoveBGInput.image`. The result is an RGBA image with the background made transparent.

In [ ]:
import time

from IPython.display import display

from remove_bg import RemoveBGInput

product_image = "https://bria-test-images.s3.us-east-1.amazonaws.com/strawberry.jpg"

task = RemoveBGInput(image=product_image)

t0 = time.time()
results = remove_bg.execute([task])
output_path = outputs_dir / "strawberry_cutout.png"
results[0].image.save(output_path)

print(f"Generated {results[0].image.size} cutout in {time.time() - t0:.1f}s")
print(f"Saved to {output_path}")
display(results[0].image)

## 5. Preview The Transparency

Alpha channels are hard to judge against a plain notebook background, so composite the cutout over a checkerboard pattern — the standard image-editor convention for "this area is transparent".

In [ ]:
import numpy as np
from PIL import Image


def checkerboard(size, tile=20, light=235, dark=200):
    w, h = size
    yy, xx = np.mgrid[0:h, 0:w]
    pattern = ((xx // tile) + (yy // tile)) % 2 == 0
    board = np.where(pattern, light, dark).astype(np.uint8)
    return Image.fromarray(np.stack([board] * 3, axis=-1))


cutout = results[0].image.convert("RGBA")
board = checkerboard(cutout.size)
preview = Image.alpha_composite(board.convert("RGBA"), cutout).convert("RGB")

preview_path = outputs_dir / "strawberry_cutout_preview.png"
preview.save(preview_path)
print(f"Saved transparency preview to {preview_path}")
display(preview)

## 6. Cleanup

Release resources held by the underlying pipeline when you are done.

In [ ]:
remove_bg.cleanup()
print("Cleanup done.")